## Diving into the details of the inclusion implementations.

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import numpy as np
from quickrules.data_handling import get_dataset
from pathlib import Path
import fuzzyroughrules.fuzzy_operators as fo
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from fuzzyroughrules.rule_induction_base import RuleGenerator
from fuzzyroughrules.approximations import LowerApproximation, MulticlassMSECVXApproximation
from fuzzyroughrules.operators import ImplicatorInclusion, OWAImplicatorInclusion
from quickrules.data_handling import test_save
from pathlib import Path
from sklearn.metrics import balanced_accuracy_score

In [57]:
name = 'ecoli' # 'bupa
nr = 7

In [58]:
x_train, y_train, t_train = get_dataset(
    Path.cwd() / 'keel-data' / f'{name}-10-fold',
    f"{nr}tra",
    get_datatypes=True
)
x_test, y_test = get_dataset(
    Path.cwd() / 'keel-data' / f'{name}-10-fold',
    f"{nr}tst"
)

In [59]:
classes = list(np.unique(np.append(y_train, y_test)))
y_train = np.array([classes.index(label) for label in y_train])
y_test = np.array([classes.index(label) for label in y_test])

In [60]:
model = RuleGenerator(
    LowerApproximation(),
    inclusion_measure=ImplicatorInclusion(),
    inclusion_threshold=0.9999,
    with_reducts=True,
)
model.fit(x_train, y_train, _)
balanced_accuracy_score(y_test, model.predict(x_test))


/Users/henri/Library/CloudStorage/OneDrive-Personal/Documents/_Work/PhD Thesis/2022-fuzzylem/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2184: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


0.7707142857142857

Comparing this to acc when using a higher threshold

In [62]:
from quickrules.weights import Weights, LinearWeights, TruncatedWeights

model_high = RuleGenerator(
    LowerApproximation(),
    # inclusion_measure=OWAImplicatorInclusion(
    #     weight_function=
    #         LinearWeights()
    # ),
    inclusion_measure=ImplicatorInclusion(),
    inclusion_threshold=1-1e-6,
    with_reducts=True,
    optimise_attribute_order=True
)
model_high.fit(x_train, y_train, t_train)
balanced_accuracy_score(y_test, model_high.predict(x_test))

/Users/henri/Library/CloudStorage/OneDrive-Personal/Documents/_Work/PhD Thesis/2022-fuzzylem/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2184: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


0.7814285714285715

In [23]:
high_credibility_predictions = []
for i in range(model_high.n_classes):
    high_credibility_predictions.append([])
for i in model_high.selected_indexes:
    high_credibility_predictions[model_high.y[i]].append(
        fo.lukasiewicz_t_norm(
            triangular_relation(
                x_test_scaled,
                model_high.X_scaled[i],
                model_high.slopes[i],
                model_high.reducts[i]
            ),
            model_high.positive_region[i]
        )
    )

In [24]:
high_credibility_predictions[6][0]

array([0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
       0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
       0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
       0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
       0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000])

In [25]:
for i in range(model_high.n_classes):
    print(len(high_credibility_predictions[i]))

7
17
2
2
9
5
1
9
